<a href="https://colab.research.google.com/github/purnimamarasinghe2005-lgtm/BI-and-Data-Mining/blob/main/diabetes_readmission_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Predicting 30-Day Hospital Readmissions for Diabetic Patients: A Business Intelligence Approach

**Module:** Business Intelligence and Data Mining (UFCFMM)  
**Assessment:** Portfolio – Group Project Portfolio  
**Students:** Sadeep Madushan,
**Date:** 02 April 2026

A business intelligence model in predicting 30-day hospital readmission of diabetic patients.

## 1. Business Problem Characterization.

Readmission of patients within 30 days of hospital stay is another issue for health care providers, which has become a major operational and economic problem. Soaring readmission rates not only add to the cost of treatment but also decrease the number of free beds and can have an overall adverse impact on performance ratings of hospitals as regarded by insurance companies and regulatory organizations. Hospitals with high readmission rates in a particular healthcare system are fined and risk their reputation.

Operationally speaking, the administrators of the hospital would need proactive intelligence to help them know about the high-risk patients ahead of the discharge to ensure that specific interventions are done towards them. The interventions can be improved discharge planning, reviewing of medication, making of follow-up calls after discharge, or even monitoring of them at home.

Diabetes is a chronic illness that is highly common in the world and which is attributed to a high rate of hospitalization and complications. The non-adherence to medication, poor glycemic control, and comorbidities may cause excessive risks of readmission. Hence, data mining in healthcare business intelligence can have a useful application in the prediction of 30-day readmission risk among patients with diabetes.

This project also seeks to come up with a predictive classification model able to determine whether a patient with diabetes is at risk of readmission within 30 days or not. The task is not to create an accurate model alone but to convert predictive outcomes into actionable information that aids in terms of hospital resource allocation and operational decision-making.

## 2. Dataset Justification

The research employs the publicly accessible UCI Machine Learning Repository of the dataset titled Diabetes 130-US hospitals in order to address this issue. The data set has more than 101,000 hospital encounters between 1999 and 2008 in 130 hospitals in the United States.

The reason why this dataset is suitable to the project is that
- It has a specific readmission outcome variable.
- It contains the demographic, diagnostic, and utilization of hospital aspects that are applicable in prediction.
- It is real-life anonymized healthcare data, which can be used in business intelligence modeling.

Nonetheless, restrictions have to be taken into account. The dataset targets specifically diabetic patients and also represents past US healthcare practices, and this may influence the generalizability. These will be the limiting factors that will be taken seriously in interpreting results.

## 3. Project Objectives

The targeted goals of this project are:

- To investigate and learn the trends related to 30-day readmissions.
- In order to construct and compare several classification models.
- To measure model performance based on the right healthcare metrics.
- To present the results of the analytical work in the form of practical strategic recommendations.
- To critically evaluate predicted healthcare analytics ethical, privacy, and security impacts.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, RocCurveDisplay, precision_recall_curve, auc
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [2]:
data_path = "/content/sample_data/Data/diabetic_data.csv"
ids_path  = "/content/sample_data/Data/IDS_mapping.csv"

df_raw = pd.read_csv(data_path)
print("Shape:", df_raw.shape)
df_raw.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/sample_data/Data/diabetic_data.csv'

In [ ]:
df_raw.info()

## Target Definition (30-day readmission)

The original `readmitted` variable has three values: `<30`, `>30`, and `NO`.  
For decision support, we define a binary target:

- **1:** readmitted within 30 days (`<30`)
- **0:** not readmitted within 30 days (`>30` or `NO`)

This makes the output directly actionable for hospital intervention planning.

In [ ]:
df = df_raw.copy()

print(df["readmitted"].value_counts())

df["readmit_30"] = (df["readmitted"] == "<30").astype(int)
print("\nBinary target counts:")
print(df["readmit_30"].value_counts())
print("\nBinary positive rate:", df["readmit_30"].mean())

## Missing values profiling

This dataset uses `?` to represent unknown values in several categorical columns.  
We first quantify `?` and then convert it to `NaN` for consistent preprocessing.

In [ ]:
missing_q = (df == "?").sum().sort_values(ascending=False)
missing_q.head(15)

In [ ]:
df = df.replace("?", np.nan)
missing_pct = df.isna().mean().sort_values(ascending=False)
missing_pct.head(15)

## Data Cleaning and Feature Selection

The dataset contains several attributes with very high missing rates. Columns with extremely high missingness provide little reliable information and may introduce noise into the model. Therefore, we apply the following decisions:

• **weight** (96.8% missing) → removed due to insufficient data  
• **max_glu_serum** (94.7% missing) → removed because most entries are unknown  
• **A1Cresult** (83.2% missing) → removed due to excessive missing values  
• **payer_code** (39.5% missing) → removed as the variable has large missingness and limited predictive value  

The column **medical_specialty** has approximately 49% missing values. However, it may still contain useful clinical information, so missing values will be treated as a separate category (“Unknown”).

Additionally, **encounter_id** is a unique identifier and does not contribute to prediction, therefore it is removed.

These decisions improve data quality while preserving clinically relevant variables for analysis.

In [ ]:
# Copy dataset
df_clean = df.copy()

# Drop columns with extremely high missing values
drop_cols = ["weight", "max_glu_serum", "A1Cresult", "payer_code", "encounter_id"]

df_clean = df_clean.drop(columns=drop_cols)

# Check remaining shape
print("New shape:", df_clean.shape)

df_clean.head()

## Handling Remaining Missing Values

Remaining missing values are relatively small.

• **race** missing values will be replaced with "Unknown"  
• **medical_specialty** missing values will be replaced with "Unknown"  
• **diag_1, diag_2, diag_3** represent diagnosis codes and missing values will be replaced with "Unknown"

This approach preserves useful information while avoiding unnecessary row removal.

In [ ]:
fill_unknown_cols = ["race", "medical_specialty", "diag_1", "diag_2", "diag_3"]

for col in fill_unknown_cols:
    if col in df_clean.columns:
        df_clean[col] = df_clean[col].fillna("Unknown")

# Check missing values again
df_clean.isna().sum().sort_values(ascending=False).head(10)

In [ ]:
df_clean["readmit_30"] = (df_clean["readmitted"] == "<30").astype(int)

df_clean["readmit_30"].value_counts()

In [ ]:
df_clean = df_clean.drop(columns=["readmitted"])

In [ ]:
df_clean.describe()

In [ ]:
df_clean.info()

In [ ]:
df_clean.shape

In [ ]:
df_clean.head()

## Exploratory Data Analysis (EDA)

Exploratory Data Analysis is conducted to understand the structure of the dataset and identify patterns related to hospital readmission. Visualisations are used to explore relationships between patient characteristics, hospital utilisation metrics, and the probability of readmission within 30 days.

In [ ]:
df_clean["readmit_30"].value_counts().plot(kind="bar")

plt.title("Distribution of 30-Day Readmission")
plt.xlabel("Readmitted within 30 days")
plt.ylabel("Number of Patients")

plt.show()

### Insight
The dataset shows a class imbalance where most patients are not readmitted within 30 days. This imbalance must be considered during model evaluation because accuracy alone may be misleading.

In [ ]:
age_readmit = df_clean.groupby("age")["readmit_30"].mean()

age_readmit.plot(kind="bar")

plt.title("Readmission Rate by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Readmission Rate")

plt.show()

### Insight
Older patient groups tend to have higher readmission rates. This suggests that elderly diabetic patients may require stronger discharge planning and follow-up care to reduce hospital readmissions.

In [ ]:
stay_readmit = df_clean.groupby("time_in_hospital")["readmit_30"].mean()

stay_readmit.plot(kind="line")

plt.title("Readmission Rate by Length of Hospital Stay")
plt.xlabel("Days in Hospital")
plt.ylabel("Readmission Rate")

plt.show()

### Insight
Patients who stay longer in the hospital tend to show a higher probability of readmission, which may indicate more severe medical conditions requiring additional monitoring after discharge.

In [ ]:
med_readmit = df_clean.groupby("num_medications")["readmit_30"].mean()

med_readmit.plot(kind="line")

plt.title("Readmission Rate by Number of Medications")
plt.xlabel("Number of Medications")
plt.ylabel("Readmission Rate")

plt.show()

### Insight
Higher medication counts are associated with increased readmission risk, possibly reflecting complex treatment plans or severe medical conditions.

In [ ]:
inpatient_readmit = df_clean.groupby("number_inpatient")["readmit_30"].mean()

inpatient_readmit.plot(kind="line")

plt.title("Readmission Rate by Previous Inpatient Visits")
plt.xlabel("Number of Previous Inpatient Visits")
plt.ylabel("Readmission Rate")

plt.show()

### Insight
Patients with multiple previous inpatient visits show significantly higher readmission rates, indicating that past hospital utilisation is a strong predictor of future readmissions.

In [ ]:
emergency_readmit = df_clean.groupby("number_emergency")["readmit_30"].mean()

emergency_readmit.plot(kind="line")

plt.title("Readmission Rate by Emergency Visits")
plt.xlabel("Number of Emergency Visits")
plt.ylabel("Readmission Rate")

plt.show()

### Insight
Frequent emergency visits are associated with higher readmission rates, suggesting that patients with unstable health conditions may require more structured follow-up care.

In [ ]:
gender_readmit = df_clean.groupby("gender")["readmit_30"].mean()

gender_readmit.plot(kind="bar")

plt.title("Readmission Rate by Gender")
plt.xlabel("Gender")
plt.ylabel("Readmission Rate")

plt.show()

### Insight
Readmission rates between genders appear relatively similar, suggesting that gender alone may not be a strong predictor of readmission risk.

In [ ]:
race_readmit = df_clean.groupby("race")["readmit_30"].mean().sort_values()

race_readmit.plot(kind="bar")

plt.title("Readmission Rate by Race")
plt.xlabel("Race")
plt.ylabel("Readmission Rate")

plt.show()

### Insight
Differences in readmission rates across racial groups highlight the importance of examining fairness and potential bias in predictive healthcare models.